# Chapter 3: VLM Architecture — Bridging Vision and Language

*Build a Multimodal Model from Scratch*

---

## Chapter Goal

Build the architecture of a **Vision-Language Model (VLM)** modeled after
LLaVA — the design used by most open-source VLMs.

After Chapter 2, we have a ViT that encodes images and a CLIP model that
aligns images with text.  But CLIP still cannot *generate text*.
To answer image questions ("What color is the cat?") we need:

1. **An image encoder** — ViT from Chapter 1 (or CLIP's image encoder).
2. **A language model decoder** — GPT from the original book.
3. **A bridge** — a small network that maps visual features into the
   GPT's token embedding space.

That bridge is the `ProjectionMLP`, and this chapter builds it.

By the end of this chapter you will understand:
1. Why a direct connection between ViT and GPT fails (the semantic gap).
2. How `ProjectionMLP` translates vision features into language features.
3. The *visual prefix* mechanism: how image tokens prepend to text tokens.
4. Loss masking: why we only supervise on text tokens, not visual ones.
5. The `set_stage1()` / `set_stage2()` interface that drives training.

In [ ]:
import os, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

os.makedirs('figures', exist_ok=True)
torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {DEVICE}')

---
## 3.1  The Semantic Gap Problem

Even after CLIP training, the ViT's patch token vectors and GPT's word
embedding vectors occupy **different coordinate systems**.

To make this concrete, let's measure the distance between the two spaces:

In [ ]:
# Simulate what ViT outputs and what GPT expects
vision_dim   = 768
language_dim = 768

# ViT patch tokens: L2-normalized, distributed on the unit hypersphere
vit_features = F.normalize(torch.randn(16, vision_dim), dim=-1)

# GPT token embeddings: learned from language, very different statistics
gpt_embeddings = torch.randn(1000, language_dim) * 0.02  # typical init scale

# Compare the two distributions
print('=== Feature Space Comparison ===')
print(f'ViT patch tokens:')
print(f'  Mean L2 norm : {vit_features.norm(dim=-1).mean():.4f}  (always 1.0 after L2 norm)')
print(f'  Value range  : [{vit_features.min():.3f}, {vit_features.max():.3f}]')
print(f'  Mean abs val : {vit_features.abs().mean():.4f}')
print()
print(f'GPT word embeddings:')
print(f'  Mean L2 norm : {gpt_embeddings.norm(dim=-1).mean():.4f}  (much smaller)')
print(f'  Value range  : [{gpt_embeddings.min():.3f}, {gpt_embeddings.max():.3f}]')
print(f'  Mean abs val : {gpt_embeddings.abs().mean():.4f}')
print()
print('The two distributions are completely different.')
print('Feeding ViT tokens directly into GPT would produce nonsense.')

### Why Does This Matter?

Imagine a GPT trained on text token embeddings where the values are all around
±0.02 (small, learned through billions of text tokens).  Now you feed it a
ViT patch token where values range from ±0.05 to ±0.5 (L2-normalized, spread
across a unit sphere).

The GPT's attention weights and layer norms are calibrated for the text
embedding distribution.  Visual tokens arriving from a completely different
distribution will:

1. Produce attention scores orders of magnitude larger or smaller than expected.
2. Cause layer normalization to behave unpredictably.
3. Result in useless or unstable activations throughout the GPT.

We need a **translation layer** between the two worlds.

---
## 3.2  ProjectionMLP: The Semantic Bridge

The `ProjectionMLP` is a two-layer MLP with a GELU activation in between.

```
ViT patch token  →  Linear(vision_dim, language_dim)  →  GELU  →  Linear(language_dim, language_dim)
```

### Why two layers instead of one?

A single linear layer can only *rescale and rotate* the feature space — it
cannot change the topology.  The hidden GELU layer allows the MLP to perform
non-linear remapping: moving points from the curved surface of the vision
embedding space to the appropriate location in the language embedding space.

In practice, this tiny MLP (< 1% of total model parameters) does most of
the heavy lifting in Stage 1 training.

In [ ]:
class ProjectionMLP(nn.Module):
    """
    Two-layer MLP that maps ViT patch tokens (vision_dim) into GPT embedding
    space (language_dim).

    This is the core bridge in LLaVA-style VLMs.  It has far fewer parameters
    than either the ViT or the GPT, but it is the most critical piece for
    enabling the two pretrained models to communicate.
    """
    def __init__(self, vision_dim, language_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(vision_dim, language_dim),
            nn.GELU(),
            nn.Linear(language_dim, language_dim),
        )

    def forward(self, x):   # x: (B, N_patches, vision_dim)
        return self.net(x)  # returns (B, N_patches, language_dim)


# Demonstrate shape transformation
vision_dim   = 128
language_dim = 256
B, N = 2, 16    # 2 images, 16 patches each

projection = ProjectionMLP(vision_dim, language_dim)
vit_output = torch.randn(B, N, vision_dim)     # from ViT
lang_tokens = projection(vit_output)           # now in language space

print(f'ViT patch tokens   : {vit_output.shape}   (B, N_patches, vision_dim)')
print(f'Projected tokens   : {lang_tokens.shape}  (B, N_patches, language_dim)')
print()
total = sum(p.numel() for p in projection.parameters())
print(f'ProjectionMLP parameters: {total:,}')

---
## 3.3  The Visual Prefix Mechanism

With the projected patch tokens now in language space, we prepend them to
the text token sequence before the GPT decoder:

```
Full input sequence:
[V_1, V_2, ..., V_N_img, t_1, t_2, ..., t_T]
  ↑ visual tokens              ↑ text tokens
  (from ViT + Projection)       (from tokenizer)
```

This is the **visual prefix** — the image becomes a prompt that the GPT
conditions all its text generation on.

The figure below shows the complete forward pass:

In [ ]:
def draw_vlm_architecture():
    from matplotlib.patches import FancyBboxPatch
    BLUE   = '#4A90D9'
    ORANGE = '#F0AD4E'
    GREEN  = '#5CB85C'
    DARK   = '#2C3E50'
    RED    = '#D9534F'
    WHITE  = '#FFFFFF'

    def _box(ax, x, y, w, h, color, text, fs=10):
        p = FancyBboxPatch((x-w/2, y-h/2), w, h,
                           boxstyle='round,pad=0.05,rounding_size=0.3',
                           facecolor=color, edgecolor=DARK, linewidth=1.2, zorder=3)
        ax.add_patch(p)
        ax.text(x, y, text, ha='center', va='center', fontsize=fs,
                color=WHITE, fontweight='bold', zorder=4)

    def _arrow(ax, x1, y1, x2, y2):
        ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                    arrowprops=dict(arrowstyle='->', color=DARK, lw=1.5), zorder=2)

    fig, ax = plt.subplots(figsize=(15, 5.5))
    ax.set_xlim(0, 15); ax.set_ylim(0, 5.5)
    ax.axis('off'); fig.patch.set_facecolor('#F8F9FA')

    img_data = np.zeros((8,8,3))
    img_data[:4,:4] = [0.3,0.6,1.0]; img_data[4:,4:] = [0.9,0.5,0.2]
    img_ax = ax.inset_axes([0.005,0.3,0.07,0.45])
    img_ax.imshow(img_data); img_ax.axis('off'); img_ax.set_title('Image', fontsize=8)

    _arrow(ax, 1.15, 2.75, 1.65, 2.75)
    _box(ax, 2.15, 2.75, 0.9, 1.4, BLUE, 'ViT\nEncoder', 9)
    ax.text(2.15, 1.75, 'N patch\ntokens', ha='center', fontsize=8, color=BLUE, style='italic')
    _arrow(ax, 2.65, 2.75, 3.15, 2.75)

    _box(ax, 3.7, 2.75, 1.0, 1.0, ORANGE, 'Proj\nMLP', 9)
    ax.text(3.7, 4.1, 'vision space\n-> language space',
            ha='center', fontsize=8, color=ORANGE, style='italic')
    _arrow(ax, 4.25, 2.75, 4.85, 2.75)

    v_tokens = 3; t_tokens = 4
    tw = 0.65; gap = 0.08; sx = 5.1

    for i in range(v_tokens):
        x = sx + i*(tw+gap)
        _box(ax, x, 2.75, tw, 0.6, BLUE, f'V{i+1}' if i < v_tokens-1 else '...', 9)

    dots_x = sx + v_tokens*(tw+gap)
    t_labels = ['<sos>', 'What', 'color', '?']
    for i, lbl in enumerate(t_labels):
        _box(ax, dots_x + i*(tw+gap), 2.75, tw, 0.6, GREEN, lbl, 8)

    seq_start = sx - tw/2 - 0.05
    seq_end   = dots_x + (len(t_labels)-1)*(tw+gap) + tw/2 + 0.05
    ax.annotate('', xy=(seq_end,2.2), xytext=(seq_start,2.2),
                arrowprops=dict(arrowstyle='-', color='#BDC3C7', lw=1.5))
    ax.text((seq_start+seq_end)/2, 1.95,
            '[ visual tokens | text tokens ]  ->  GPT Decoder',
            ha='center', fontsize=9.5, color=DARK,
            bbox=dict(facecolor='white', edgecolor='#BDC3C7', boxstyle='round'))

    _arrow(ax, seq_end+0.1, 2.75, seq_end+0.6, 2.75)
    _box(ax, seq_end+1.15, 2.75, 1.0, 1.2, GREEN, 'GPT\nDecoder', 9)
    _arrow(ax, seq_end+1.65, 2.75, seq_end+2.15, 2.75)
    ax.text(seq_end+2.7, 2.75, '"Blue"',
            ha='center', va='center', fontsize=12, fontweight='bold', color=GREEN,
            bbox=dict(facecolor='#EAFAF1', edgecolor=GREEN, boxstyle='round', lw=1.5))

    ax.text(6.5, 4.55,
            'Loss computed on text tokens only (visual tokens masked with -100)',
            ha='center', fontsize=9, color=RED, style='italic',
            bbox=dict(facecolor='#FDEDEC', edgecolor=RED, boxstyle='round', lw=1))

    ax.set_title('VLM Forward Pass: Visual Tokens as a Prefix to the Language Model',
                 fontsize=12, fontweight='bold', color=DARK)
    plt.tight_layout()
    plt.savefig('figures/ch03_vlm_arch.png', dpi=120, bbox_inches='tight')
    plt.show()

draw_vlm_architecture()

---
## 3.4  Loss Masking: Supervise Text, Ignore Visual Tokens

When training, we compute next-token-prediction loss on the combined sequence:
```
[V_1, ..., V_N, t_1, t_2, ..., t_T]
```

But we must **only compute loss on text positions**, not visual ones.

**Why?**  The visual tokens were produced by the ViT — they are not word IDs.
There is no "correct next visual token" to predict.  Asking the model to
predict the next visual patch embedding would be meaningless and would corrupt
the text generation capability.

The implementation uses PyTorch's `ignore_index=-100` in `F.cross_entropy`:

```python
labels = input_ids.clone()
labels[:, :n_visual] = -100    # mask out visual positions
loss = F.cross_entropy(logits, labels.flatten(), ignore_index=-100)
```

In [ ]:
# Demonstrate loss masking concretely
B      = 2
n_img  = 4    # visual token positions
T      = 6    # text token positions
vocab  = 20

# Simulate model logits for the combined sequence
logits = torch.randn(B, n_img + T, vocab)

# Ground truth token IDs (text only)
input_ids = torch.randint(0, vocab, (B, T))

# Build labels: -100 for visual positions, real token IDs for text positions
# Next-token prediction: labels[t] = input_ids[t+1]
# We predict the next text token from each text position
text_logits  = logits[:, n_img:-1, :]     # (B, T-1, vocab)  text positions
text_targets = input_ids[:, 1:]           # (B, T-1)          shifted targets

loss_without_masking = F.cross_entropy(
    text_logits.reshape(-1, vocab),
    text_targets.reshape(-1)
)

# With masking: mark some positions as -100
masked_targets = text_targets.clone()
masked_targets[:, :2] = -100   # pretend first 2 text tokens are "instruction" (not supervised)

loss_with_masking = F.cross_entropy(
    text_logits.reshape(-1, vocab),
    masked_targets.reshape(-1),
    ignore_index=-100
)

print(f'Loss without masking (all text): {loss_without_masking:.4f}')
print(f'Loss with masking (answer only): {loss_with_masking:.4f}')
print()
print('ignore_index=-100: positions labeled -100 contribute ZERO gradient.')
print('This is how the model learns to generate responses, not echo the prompt.')

---
## 3.5  GPT Decoder with Visual Prefix Support

We extend the GPT decoder from the original book with one new parameter:
`visual_prefix`.  When provided, the visual tokens are prepended to the
text tokens before attention.

**Attention mask with visual prefix:**
* Visual tokens attend to each other **bidirectionally** (they are all
  simultaneously available; there is no temporal order among patches).
* Text tokens attend **causally** to all previous positions — including
  all visual tokens.

This combines two attention patterns in one forward pass:

```
          V_1  V_2  V_3  t_1  t_2  t_3
  V_1   [ OK   OK   OK   X    X    X  ]
  V_2   [ OK   OK   OK   X    X    X  ]
  V_3   [ OK   OK   OK   X    X    X  ]
  t_1   [ OK   OK   OK   OK   X    X  ]
  t_2   [ OK   OK   OK   OK   OK   X  ]
  t_3   [ OK   OK   OK   OK   OK   OK ]
         ^-- all visual visible --^  ^-- causal text --^
```

In [ ]:
class GPTBlock(nn.Module):
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.attn  = nn.MultiheadAttention(embed_dim, n_heads, batch_first=True)
        self.ffn   = nn.Sequential(
            nn.Linear(embed_dim, embed_dim*4), nn.GELU(),
            nn.Linear(embed_dim*4, embed_dim))

    def forward(self, x, n_visual=0):
        T = x.shape[1]
        # Build combined attention mask
        causal = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        if n_visual > 0:
            # Visual rows: no restriction
            causal[:n_visual, :]    = False
            # Text rows: can see all visual tokens
            causal[n_visual:, :n_visual] = False
        h = self.norm1(x)
        out, _ = self.attn(h, h, h, attn_mask=causal)
        x = x + out
        return x + self.ffn(self.norm2(x))


class GPTDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, depth=4, n_heads=4, max_seq=256):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed   = nn.Embedding(max_seq, embed_dim)
        self.blocks      = nn.ModuleList([GPTBlock(embed_dim, n_heads) for _ in range(depth)])
        self.norm        = nn.LayerNorm(embed_dim)
        self.lm_head     = nn.Linear(embed_dim, vocab_size, bias=False)
        # Weight tying: share token embedding and output projection matrices
        # This halves parameters in the embedding+output layers and improves performance.
        self.lm_head.weight = self.token_embed.weight
        self.max_seq = max_seq

    def forward(self, input_ids, visual_prefix=None):
        B, T    = input_ids.shape
        device  = input_ids.device
        n_visual = 0

        if visual_prefix is not None:
            n_visual = visual_prefix.shape[1]
            # Visual tokens: positions 0..n_visual-1
            vpos  = self.pos_embed(torch.arange(n_visual, device=device)).unsqueeze(0)
            # Text tokens: positions n_visual..n_visual+T-1
            tpos  = self.pos_embed(
                torch.arange(n_visual, n_visual+T, device=device)).unsqueeze(0)
            x = torch.cat([visual_prefix + vpos,
                            self.token_embed(input_ids) + tpos], dim=1)
        else:
            pos = self.pos_embed(torch.arange(T, device=device)).unsqueeze(0)
            x   = self.token_embed(input_ids) + pos

        for blk in self.blocks:
            x = blk(x, n_visual=n_visual)
        x = self.norm(x)
        return self.lm_head(x), n_visual   # (B, n_visual+T, vocab), n_visual

print('GPTBlock, GPTDecoder defined.')

---
## 3.6  Assembling the Complete VLM

We now connect all three components: ViT → ProjectionMLP → GPT.

In [ ]:
# ── Minimal ViT encoder ─────────────────────────────────────────────────────

class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
    def forward(self, x): return self.proj(x).flatten(2).transpose(1, 2)

class PositionalEmbedding(nn.Module):
    def __init__(self, n_patches, embed_dim):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches+1, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
    def forward(self, x): return x + self.pos_embed

class BidirectionalMHA(nn.Module):
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim); self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(nn.Linear(embed_dim, embed_dim*4), nn.GELU(),
                                 nn.Linear(embed_dim*4, embed_dim))
    def forward(self, x):
        h = self.norm1(x); out, _ = self.attn(h, h, h); x = x + out
        return x + self.ffn(self.norm2(x))

class ViTEncoder(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim, depth, n_heads):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        n_p = self.patch_embed.n_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = PositionalEmbedding(n_p, embed_dim)
        self.blocks = nn.ModuleList([BidirectionalMHA(embed_dim, n_heads) for _ in range(depth)])
        self.norm   = nn.LayerNorm(embed_dim)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
    def forward(self, x):
        B = x.shape[0]
        t = torch.cat([self.cls_token.expand(B,-1,-1), self.patch_embed(x)], dim=1)
        t = self.pos_embed(t)
        for blk in self.blocks: t = blk(t)
        return self.norm(t)[:, 1:, :]   # return patch tokens (exclude CLS)

print('ViTEncoder defined.')

In [ ]:
class VisionLanguageModel(nn.Module):
    def __init__(self, img_size=32, patch_size=8,
                 vision_dim=128, vision_depth=3, vision_heads=4,
                 language_dim=128, language_depth=4, language_heads=4,
                 vocab_size=500, max_seq=64):
        super().__init__()
        self.vit        = ViTEncoder(img_size, patch_size, 3,
                                     vision_dim, vision_depth, vision_heads)
        self.projection = ProjectionMLP(vision_dim, language_dim)
        self.gpt        = GPTDecoder(vocab_size, language_dim,
                                     language_depth, language_heads, max_seq)

    def forward(self, images, input_ids, labels=None):
        # 1. Image -> patch tokens (ViT)
        patch_tokens  = self.vit(images)                   # (B, N_img, vision_dim)
        # 2. Project to language space
        visual_prefix = self.projection(patch_tokens)      # (B, N_img, language_dim)
        # 3. GPT with visual prefix
        logits, n_vis = self.gpt(input_ids, visual_prefix=visual_prefix)
        # logits: (B, n_vis + T, vocab_size)

        loss = None
        if labels is not None:
            # Only supervise text positions
            text_logits  = logits[:, n_vis:-1, :]   # predict each text token
            text_targets = labels[:, 1:]             # shifted by 1 (next token)
            loss = F.cross_entropy(
                text_logits.reshape(-1, text_logits.size(-1)),
                text_targets.reshape(-1),
                ignore_index=-100)
        return logits, loss

    def set_stage1(self):
        """Freeze ViT and GPT; train only the ProjectionMLP."""
        for p in self.vit.parameters():        p.requires_grad_(False)
        for p in self.gpt.parameters():        p.requires_grad_(False)
        for p in self.projection.parameters(): p.requires_grad_(True)

    def set_stage2(self):
        """Freeze ViT only; train ProjectionMLP and GPT."""
        for p in self.vit.parameters():        p.requires_grad_(False)
        for p in self.projection.parameters(): p.requires_grad_(True)
        for p in self.gpt.parameters():        p.requires_grad_(True)

    def count_params(self, trainable_only=False):
        params = [p for p in self.parameters() if p.requires_grad or not trainable_only]
        n = sum(p.numel() for p in params)
        return n

print('VisionLanguageModel defined.')

---
## 3.7  Verifying the Forward Pass

Before training, let's verify shapes at each step of the forward pass:

In [ ]:
vlm = VisionLanguageModel(
    img_size=32, patch_size=8,
    vision_dim=128, vision_depth=3, vision_heads=4,
    language_dim=128, language_depth=4, language_heads=4,
    vocab_size=500, max_seq=64,
).to(DEVICE)

B = 2
images    = torch.randn(B, 3, 32, 32).to(DEVICE)
input_ids = torch.randint(0, 500, (B, 10)).to(DEVICE)
labels    = input_ids.clone()

# Step-by-step forward pass
with torch.no_grad():
    patch_tokens  = vlm.vit(images)
    visual_prefix = vlm.projection(patch_tokens)
    logits, n_vis = vlm.gpt(input_ids, visual_prefix=visual_prefix)

print(f'Images          : {images.shape}')
print(f'Patch tokens    : {patch_tokens.shape}   (B, N_patches, vision_dim)')
print(f'Visual prefix   : {visual_prefix.shape}  (B, N_patches, language_dim)')
print(f'n_visual        : {n_vis}  (patch tokens in GPT sequence)')
print(f'GPT logits      : {logits.shape}   (B, n_vis+T, vocab_size)')
print()
print(f'Text logits     : {logits[:, n_vis:-1, :].shape}  (predict each text token)')
print(f'Text targets    : {labels[:, 1:].shape}           (shifted by 1)')

# Full forward with loss
logits, loss = vlm(images, input_ids, labels=labels)
print(f'\nFull forward loss: {loss.item():.4f}')

---
## 3.8  Parameter Distribution by Component

In [ ]:
def plot_parameter_distribution(vlm):
    comps = {
        'ViT\nEncoder':   sum(p.numel() for p in vlm.vit.parameters()),
        'Projection\nMLP': sum(p.numel() for p in vlm.projection.parameters()),
        'GPT\nDecoder':   sum(p.numel() for p in vlm.gpt.parameters()),
    }
    total = sum(comps.values())
    colors = ['#4A90D9', '#F0AD4E', '#5CB85C']

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

    for ax_idx, (stage_name, trainable_set) in enumerate([
        ('All Parameters',    set(comps)),
        ('Stage 1 Trainable', {'Projection\nMLP'}),
        ('Stage 2 Trainable', {'Projection\nMLP', 'GPT\nDecoder'}),
    ]):
        ax = axes[ax_idx]
        for i, (name, count) in enumerate(comps.items()):
            pct = 100 * count / total
            is_train = name in trainable_set
            ax.bar(name, count, color=colors[i],
                   alpha=1.0 if is_train else 0.25,
                   edgecolor='black' if is_train else 'grey',
                   linewidth=1.5 if is_train else 0.5)
            lbl = f'{count:,}\n({pct:.1f}%)'
            if not is_train: lbl += '\n[frozen]'
            ax.text(i, count + total*0.01, lbl,
                    ha='center', va='bottom', fontsize=7.5)
        ax.set_title(stage_name, fontsize=11, fontweight='bold')
        ax.set_ylim(0, total * 1.4)

    plt.suptitle('VLM Parameter Distribution — Frozen vs Trainable by Stage',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/ch03_param_dist.png', dpi=120, bbox_inches='tight')
    plt.show()

    proj = comps['Projection\nMLP']
    gpt  = comps['GPT\nDecoder']
    print(f'Stage 1: train {proj:,} / {total:,} params = {100*proj/total:.1f}%')
    print(f'Stage 2: train {proj+gpt:,} / {total:,} params = {100*(proj+gpt)/total:.1f}%')

plot_parameter_distribution(vlm)

---
## 3.9  Chapter Summary

| Concept | Key Insight |
|---------|-------------|
| **Semantic gap** | ViT and GPT features are in different coordinate systems; they cannot be directly concatenated |
| **ProjectionMLP** | 2-layer MLP with GELU; non-linear alignment between vision and language spaces |
| **Visual prefix** | Projected patch tokens prepend to text tokens; GPT conditions on both |
| **Attention mask** | Visual tokens: bidirectional; text tokens: causal + can see all visual |
| **Loss masking** | Labels for visual positions = -100; `F.cross_entropy(ignore_index=-100)` skips them |
| **Weight tying** | `lm_head.weight = token_embed.weight`; same matrix for input and output |
| **Stage interface** | `set_stage1()` / `set_stage2()` control which components receive gradients |

The VLM is now fully assembled.  Chapter 4 explains *how to train it* —
which requires a two-stage protocol that prevents catastrophic forgetting
of the pretrained ViT and GPT knowledge.